# 两轮自平衡小车

## 项目简介

本项目实现一个基于 Arduino 或兼容板、MPU6050 和双电机驱动模块的两轮自平衡小车控制原型。系统以俯仰角为核心反馈量，通过固定周期 PID 运算输出左右电机的同步补偿指令，使车体尽量保持直立，并在倾角过大时主动停机保护。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| 主控板 | 1 | 负责平衡控制运算 |
| MPU6050 | 1 | 提供姿态角反馈 |
| 双路电机驱动模块 | 1 | 驱动左右轮 |
| 直流减速电机 | 2 | 作为小车驱动轮 |
| 电池与稳压模块 | 1 组 | 提供主控与电机电源 |


## 核心功能

- 通过 I2C 读取 MPU6050 姿态角。
- 按 10ms 固定控制周期执行平衡 PID 计算。
- 将控制量映射为双电机同向输出，实现基础扶正。
- 倾角超过安全阈值时立即停机并清空积分项。
- 通过串口输出角度与控制量，便于整定参数。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 左电机 PWM | D5 |
| 左电机方向 | D6、D7 |
| 右电机 PWM | D9 |
| 右电机方向 | D10、D11 |
| MPU6050 SDA / SCL | I2C 总线 |


## 接线说明

- MPU6050 接入主控板 I2C 接口，同时共地。
- 左右电机分别接电机驱动的 A、B 通道，PWM 引脚接驱动 EN 端，方向引脚接 IN1/IN2。
- 电机供电建议独立于主控逻辑电源，并确保地线共地，否则姿态数据和驱动输出会不稳定。
- 初次测试时建议把车体架空，先观察角度输出和电机方向是否一致，再落地调参。


## 运行方式

1. 打开 `src/two_wheel_self_balancing_car/two_wheel_self_balancing_car.ino`。
2. 安装 `MPU6050_tockn` 库。
3. 完成 MPU6050、电机驱动和电源接线。
4. 上传程序后执行陀螺仪零偏校准。
5. 先架空测试，再逐步调节 `kp`、`ki`、`kd`。


## 控制逻辑说明

- 主循环先判断是否达到控制周期，未到时间就直接返回，保证控制节奏稳定。
- 每次控制先更新 MPU6050 姿态角，并取俯仰角作为平衡误差输入。
- 若倾角超过 `SAFE_ANGLE_LIMIT`，则立即停机并清空积分项，避免倒地后持续加力。
- 正常情况下计算比例、积分、微分三项，合成 `motorCommand`。
- 最终把正负控制量映射为前后方向，把绝对值映射为左右轮 PWM。


## 调试建议

- 先只调 `kp`，观察车体是否具备明显扶正趋势。
- 再加少量 `kd` 抑制来回振荡。
- `ki` 建议最后少量加入，用于修正静态偏差，过大容易积分饱和。
- 如果车体一启动就朝反方向冲，先检查姿态角正负方向和电机前后方向是否匹配。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 姿态读取 | 串口能连续输出合理俯仰角 |
| PID 输出 | 车体前倾和后仰时 PWM 方向会反向变化 |
| 安全保护 | 倾角明显过大时电机立即停止 |
| 参数整定 | 调整 PID 参数后输出变化明显可观察 |


## 可扩展方向

- 增加编码器速度环与位置环。
- 增加蓝牙遥控或上位机调参界面。
- 增加电池电压检测与低压保护。
- 将基础 PID 升级为串级控制结构。
